# Real Madrid — Phase 4.16 PostgreSQL Deployment (Google Colab)

This notebook deploys the existing Phase 4.16 package to a PostgreSQL 13+ server.

**Recommended route for a restricted PC:** hosted PostgreSQL (for example Neon) + Google Colab as the client.

## Before running
1. Create a PostgreSQL project/database with your provider.
2. For Neon, use the **direct** connection string from the Console's **Connect** dialog (turn connection pooling off for this migration/load task). The connection string is a secret.
3. Download `Real_Madrid_Master_Datasets_Phase_4_16_2026-11-19.zip` from this conversation.
4. Run the cells from top to bottom and upload that ZIP when prompted.

The notebook asks for credentials using hidden input. **Do not paste passwords or connection strings into chat, notebook text cells, or screenshots.**

The database creation cell is written for a Neon-style admin connection to the default database (usually `neondb`). If your provider has already created the target database or does not permit `CREATE DATABASE`, use the alternative in that cell.

**Do not run the optional `08_reset_schemas.sql` script.** It is destructive.

In [5]:

!apt-get update -qq
!apt-get install -y postgresql-client

import shutil

psql_path = shutil.which("psql")

if psql_path:
    print("SUCCESS: PostgreSQL client installed")
    print("PSQL location:", psql_path)
else:
    raise RuntimeError("PSQL installation failed")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  postgresql-client-16 postgresql-client-common
Suggested packages:
  postgresql-16 postgresql-doc-16
The following NEW packages will be installed:
  postgresql-client postgresql-client-16 postgresql-client-common
0 upgraded, 3 newly installed, 0 to remove and 55 not upgraded.
Need to get 1,349 kB of archives.
After this operation, 4,526 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 postgresql-client-common all 257build1.1 [36.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 postgresql-client-16 amd64 16.15-0ubuntu0.24.04.1 [1,301 kB]
Get:3 http://archive.ubuntu.com/ubuntu 

## 1. Install the PostgreSQL client and Python driver

Colab will install the client tools into its temporary runtime. Nothing needs to be installed on your Windows PC.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y postgresql-client
!pip -q install psycopg2-binary

## 2. Upload and verify the Phase 4.16 ZIP

Upload the exact file named `Real_Madrid_Master_Datasets_Phase_4_16_2026-11-19.zip`.
The archive is checked before extraction.

In [2]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Please upload exactly one Phase 4.16 ZIP. ZIP files detected: {zip_names}")

ZIP_PATH = Path("/content") / zip_names[0]
EXPECTED_ZIP = "Real_Madrid_Master_Datasets_Phase_4_16_2026-11-19.zip"
if ZIP_PATH.name != EXPECTED_ZIP:
    raise ValueError(f"Wrong archive. Expected {EXPECTED_ZIP}; received {ZIP_PATH.name}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    bad_member = zf.testzip()
    if bad_member:
        raise RuntimeError(f"ZIP integrity check failed at: {bad_member}")
    names = zf.namelist()
    required = [
        "Real_Madrid_Phase_4_16/README_Phase_4_16.md",
        "Real_Madrid_Phase_4_16/sql/01_create_schemas.sql",
        "Real_Madrid_Phase_4_16/sql/02_create_raw_staging.sql",
        "Real_Madrid_Phase_4_16/sql/03_load_raw_csvs.psql",
        "Real_Madrid_Phase_4_16/sql/04_create_core_tables.sql",
        "Real_Madrid_Phase_4_16/sql/05_load_core.sql",
        "Real_Madrid_Phase_4_16/sql/06_validate.sql",
        "Real_Madrid_Phase_4_16/sql/07_create_views.sql",
    ]
    missing = [p for p in required if p not in names]
    if missing:
        raise FileNotFoundError(f"Required Phase 4.16 files missing from ZIP: {missing}")
    EXTRACT_DIR = Path("/content/real_madrid_phase416")
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True)
    zf.extractall(EXTRACT_DIR)

WORK_DIR = EXTRACT_DIR
PHASE_DIR = WORK_DIR / "Real_Madrid_Phase_4_16"
SQL_DIR = PHASE_DIR / "sql"
DATA_DIR = PHASE_DIR / "data"

print("ZIP integrity: PASS")
print("Archive entries:", len(names))
print("Phase 4.16 data CSV files:", len(list(DATA_DIR.glob("*.csv"))))
print("Working directory:", WORK_DIR)
print("README:", PHASE_DIR / "README_Phase_4_16.md")

Saving Real_Madrid_Master_Datasets_Phase_4_16_2026-11-19.zip to Real_Madrid_Master_Datasets_Phase_4_16_2026-11-19.zip
ZIP integrity: PASS
Archive entries: 104
Phase 4.16 data CSV files: 21
Working directory: /content/real_madrid_phase416
README: /content/real_madrid_phase416/Real_Madrid_Phase_4_16/README_Phase_4_16.md


## 3. Connect to PostgreSQL and create the target database

### If using Neon
- In Neon Console, open your project and click **Connect**.
- Select the branch, database (usually `neondb`) and role.
- Turn **Connection pooling off** to copy the direct connection string.
- Paste it into the hidden prompt below.

This cell checks whether `real_madrid_analytics` exists and creates it if needed. It does not print the connection string.

### If your provider already created `real_madrid_analytics`
Set `CREATE_DATABASE = False` in the code and paste the direct connection string for that database when prompted.

If your provider does not grant database-creation permission, create `real_madrid_analytics` in its dashboard first, then use the second option.

In [3]:
from getpass import getpass
from urllib.parse import urlsplit, urlunsplit
import psycopg2
from psycopg2 import sql

TARGET_DB = "real_madrid_analytics"
CREATE_DATABASE = True  # Change to False if the provider already created the target database.

if CREATE_DATABASE:
    ADMIN_DSN = getpass("Paste the DIRECT admin connection string to the provider's default/admin database: ")
    if not ADMIN_DSN.startswith(("postgresql://", "postgres://")):
        raise ValueError("Connection string should start with postgresql:// or postgres://")

    admin_conn = psycopg2.connect(ADMIN_DSN)
    admin_conn.autocommit = True
    try:
        with admin_conn.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (TARGET_DB,))
            exists = cur.fetchone() is not None
            if not exists:
                cur.execute(
                    sql.SQL("CREATE DATABASE {} WITH ENCODING 'UTF8' TEMPLATE template0")
                    .format(sql.Identifier(TARGET_DB))
                )
                print(f"Created database: {TARGET_DB}")
            else:
                print(f"Database already exists; leaving it in place: {TARGET_DB}")
    finally:
        admin_conn.close()

    parts = urlsplit(ADMIN_DSN)
    TARGET_DSN = urlunsplit((parts.scheme, parts.netloc, "/" + TARGET_DB, parts.query, parts.fragment))
else:
    TARGET_DSN = getpass("Paste the DIRECT connection string for real_madrid_analytics: ")
    if not TARGET_DSN.startswith(("postgresql://", "postgres://")):
        raise ValueError("Connection string should start with postgresql:// or postgres://")

if "-pooler" in urlsplit(TARGET_DSN).hostname if urlsplit(TARGET_DSN).hostname else False:
    print("WARNING: this appears to be a pooled endpoint. Prefer the direct endpoint for schema deployment.")

test_conn = psycopg2.connect(TARGET_DSN)
try:
    with test_conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user, version()")
        db_name, db_user, pg_version = cur.fetchone()
    print("Connection: SUCCESS")
    print("Database:", db_name)
    print("Role:", db_user)
    print("Server:", pg_version.split(",")[0])
    if db_name != TARGET_DB:
        raise RuntimeError(f"Connected to {db_name}, expected {TARGET_DB}")
finally:
    test_conn.close()

Paste the DIRECT admin connection string to the provider's default/admin database: ··········
Created database: real_madrid_analytics
Connection: SUCCESS
Database: real_madrid_analytics
Role: neondb_owner
Server: PostgreSQL 18.7 (84da7d8) on aarch64-unknown-linux-gnu


## 4. Run the Phase 4.16 deployment scripts in order

The package's `00_create_database.sql` is handled by the previous cell so database creation is safe to repeat. The remaining scripts run in the documented order.

The raw import script uses `psql` client-side `\\copy`; the working directory is set to the extracted package root so its relative CSV paths resolve correctly.

If any script fails, the notebook stops and displays the PostgreSQL error. Do not continue to later scripts until the error is reviewed.

In [6]:
import subprocess

def run_psql(script_name):
    script_path = SQL_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(script_path)

    print("\\n" + "=" * 78)
    print("RUNNING:", script_name)
    print("=" * 78)

    result = subprocess.run(
        [
            "psql",
            TARGET_DSN,
            "-X",
            "-v", "ON_ERROR_STOP=1",
            "-f", str(script_path)
        ],
        cwd=str(WORK_DIR),
        text=True,
        capture_output=True
    )

    if result.stdout:
        print(result.stdout[-12000:])
    if result.stderr:
        print(result.stderr[-12000:])

    if result.returncode != 0:
        raise RuntimeError(
            f"{script_name} FAILED with exit code {result.returncode}. "
            "Stop here and review the error before retrying."
        )
    print("STATUS: SUCCESS")

deployment_scripts = [
    "01_create_schemas.sql",
    "02_create_raw_staging.sql",
    "03_load_raw_csvs.psql",
    "04_create_core_tables.sql",
    "05_load_core.sql",
    "06_validate.sql",
    "07_create_views.sql",
]

for script in deployment_scripts:
    run_psql(script)

print("\\nALL PHASE 4.16 DEPLOYMENT SCRIPTS COMPLETED.")

\n==============================================================================
RUNNING: 01_create_schemas.sql
CREATE SCHEMA
CREATE SCHEMA
CREATE SCHEMA
CREATE TABLE

STATUS: SUCCESS
\n==============================================================================
RUNNING: 02_create_raw_staging.sql
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE

STATUS: SUCCESS
\n==============================================================================
RUNNING: 03_load_raw_csvs.psql
TRUNCATE TABLE
COPY 313
COPY 11
COPY 47
COPY 32
COPY 11
COPY 27
COPY 27
COPY 157
COPY 159
COPY 631
COPY 204
COPY 11
COPY 631
COPY 343
COPY 157
COPY 0
COPY 386
COPY 214
COPY 23
COPY 41
COPY 23

STATUS: SUCCESS
\n==============================================================================
RUNNING:

RuntimeError: 05_load_core.sql FAILED with exit code 3. Stop here and review the error before retrying.

In [7]:

from pathlib import Path

sql_path = SQL_DIR / "05_load_core.sql"

sql = sql_path.read_text(encoding="utf-8")

old = """INSERT INTO core.dim_season (season, season_role, is_baseline)
SELECT season, COALESCE(MAX(season_role), 'Main'), BOOL_OR(season_role = 'Baseline')
FROM ("""

new = """INSERT INTO core.dim_season (season, season_role, is_baseline)
SELECT s.season, COALESCE(MAX(s.season_role), 'Main'), BOOL_OR(s.season_role = 'Baseline')
FROM ("""

if old not in sql:
    raise ValueError("Original query not found. Stop and show me the file.")

sql = sql.replace(old, new, 1)

old_end = """WHERE season IS NOT NULL
GROUP BY season;"""

new_end = """WHERE s.season IS NOT NULL
GROUP BY s.season;"""

if old_end not in sql:
    raise ValueError("Original WHERE/GROUP BY not found.")

sql = sql.replace(old_end, new_end, 1)

sql_path.write_text(sql, encoding="utf-8")

print("SUCCESS: SQL correction applied.")
print("Updated file:", sql_path)


SUCCESS: SQL correction applied.
Updated file: /content/real_madrid_phase416/Real_Madrid_Phase_4_16/sql/05_load_core.sql


In [9]:

from pathlib import Path
import re

sql_path = SQL_DIR / "05_load_core.sql"
sql = sql_path.read_text(encoding="utf-8")

pattern = (
    r'INSERT INTO core\.dim_season \(season, season_role, is_baseline\)'
    r'\nSELECT .*?\nGROUP BY s\.season;'
)

replacement = """INSERT INTO core.dim_season (season, season_role, is_baseline)
SELECT s.season,
       COALESCE(MAX(s.season_role), 'Main'),
       BOOL_OR(s.season_role = 'Baseline')
FROM (
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_all_matches AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_player_season_stats AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_player_competition_stats AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_transfers AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_trophies AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_la_liga_standings AS r
) AS s
WHERE s.season IS NOT NULL
GROUP BY s.season;"""

updated, count = re.subn(
    pattern, replacement, sql, count=1, flags=re.S
)

if count == 1:
    sql_path.write_text(updated, encoding="utf-8")
    print("SUCCESS: Corrected SQL block saved.")
elif "FROM raw.real_madrid_all_matches AS r" in sql:
    print("SQL block is already corrected.")
else:
    raise RuntimeError("Expected SQL block not found. Stop here.")

print("File:", sql_path)


SUCCESS: Corrected SQL block saved.
File: /content/real_madrid_phase416/Real_Madrid_Phase_4_16/sql/05_load_core.sql


In [10]:
run_psql("05_load_core.sql")

\n==============================================================================
RUNNING: 05_load_core.sql
BEGIN
TRUNCATE TABLE

psql:/content/real_madrid_phase416/Real_Madrid_Phase_4_16/sql/05_load_core.sql:31: ERROR:  column r.season_role does not exist
LINE 15: ...  SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_...
                                                              ^



RuntimeError: 05_load_core.sql FAILED with exit code 3. Stop here and review the error before retrying.

In [11]:

from pathlib import Path

sql_path = SQL_DIR / "05_load_core.sql"
sql = sql_path.read_text(encoding="utf-8")

start_marker = "INSERT INTO core.dim_season (season, season_role, is_baseline)"
next_marker = "\n\nINSERT INTO core.dim_competition"

start = sql.index(start_marker)
end = sql.index(next_marker, start)

corrected_block = """INSERT INTO core.dim_season (season, season_role, is_baseline)
SELECT s.season,
       COALESCE(MAX(s.season_role), 'Main'),
       BOOL_OR(s.season_role = 'Baseline')
FROM (
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_all_matches AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_player_season_stats AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_player_competition_stats AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULL::TEXT AS season_role
    FROM raw.real_madrid_transfers AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_trophies AS r
    UNION ALL
    SELECT NULLIF(r."season", '') AS season, NULLIF(r."season_role", '') AS season_role
    FROM raw.real_madrid_la_liga_standings AS r
) AS s
WHERE s.season IS NOT NULL
GROUP BY s.season;"""

sql = sql[:start] + corrected_block + sql[end:]
sql_path.write_text(sql, encoding="utf-8")

print("SUCCESS: Definitive SQL correction saved.")

print("\nTransfers handling:")
print("Season: preserved")
print("Season role: NULL (not present in source)")


SUCCESS: Definitive SQL correction saved.

Transfers handling:
Season: preserved
Season role: NULL (not present in source)


In [12]:
run_psql("05_load_core.sql")

\n==============================================================================
RUNNING: 05_load_core.sql
BEGIN
TRUNCATE TABLE
INSERT 0 11
INSERT 0 7
INSERT 0 39
INSERT 0 157
INSERT 0 631
INSERT 0 631
INSERT 0 386
INSERT 0 343
INSERT 0 0
INSERT 0 214
INSERT 0 204
INSERT 0 23
INSERT 0 11
INSERT 0 159
INSERT 0 157
INSERT 0 23
INSERT 0 11
INSERT 0 41
INSERT 0 47
INSERT 0 32
INSERT 0 313
INSERT 0 11
INSERT 0 27
INSERT 0 27
COMMIT
INSERT 0 1

STATUS: SUCCESS


In [13]:
run_psql("06_validate.sql")

\n==============================================================================
RUNNING: 06_validate.sql
DO
          table_name           | rows 
-------------------------------+------
 dim_player                    |  157
 fact_financials               |  204
 fact_laliga_standings         |   11
 fact_match_team_stats         |  631
 fact_matches                  |  631
 fact_player_competition_stats |  343
 fact_player_match_stats       |    0
 fact_player_season_stats      |  386
 fact_transfers                |  214
 fact_trophies                 |   23
 source_register               |   41
(11 rows)


STATUS: SUCCESS


In [14]:
run_psql("07_create_views.sql")

\n==============================================================================
RUNNING: 07_create_views.sql
CREATE VIEW
CREATE VIEW
CREATE VIEW
CREATE VIEW

STATUS: SUCCESS


## 5. Verify the database row counts

The following query checks the principal curated tables against the Phase 4.16 release baseline. The SQL validation script (`06_validate.sql`) has already raised an error if its assertions fail; this cell gives a readable count summary as a second check.

In [15]:
import pandas as pd

count_sql = '''
SELECT 'fact_matches' AS table_name, COUNT(*)::BIGINT AS actual_rows, 631::BIGINT AS expected_rows FROM core.fact_matches
UNION ALL SELECT 'fact_match_team_stats', COUNT(*)::BIGINT, 631::BIGINT FROM core.fact_match_team_stats
UNION ALL SELECT 'fact_player_season_stats', COUNT(*)::BIGINT, 386::BIGINT FROM core.fact_player_season_stats
UNION ALL SELECT 'fact_player_competition_stats', COUNT(*)::BIGINT, 343::BIGINT FROM core.fact_player_competition_stats
UNION ALL SELECT 'fact_player_match_stats', COUNT(*)::BIGINT, 0::BIGINT FROM core.fact_player_match_stats
UNION ALL SELECT 'fact_transfers', COUNT(*)::BIGINT, 214::BIGINT FROM core.fact_transfers
UNION ALL SELECT 'fact_financials', COUNT(*)::BIGINT, 204::BIGINT FROM core.fact_financials
UNION ALL SELECT 'fact_trophies', COUNT(*)::BIGINT, 23::BIGINT FROM core.fact_trophies
UNION ALL SELECT 'fact_laliga_standings', COUNT(*)::BIGINT, 11::BIGINT FROM core.fact_laliga_standings
UNION ALL SELECT 'dim_player', COUNT(*)::BIGINT, 157::BIGINT FROM core.dim_player
UNION ALL SELECT 'player_name_alias_crosswalk', COUNT(*)::BIGINT, 159::BIGINT FROM core.player_name_alias_crosswalk
ORDER BY table_name;
'''

conn = psycopg2.connect(TARGET_DSN)
try:
    count_df = pd.read_sql_query(count_sql, conn)
finally:
    conn.close()

count_df["status"] = count_df.apply(
    lambda r: "PASS" if r["actual_rows"] == r["expected_rows"] else "FAIL",
    axis=1
)
display(count_df)
assert (count_df["status"] == "PASS").all(), "One or more database row counts do not match the Phase 4.16 baseline."
print("DATABASE ROW-COUNT CHECK: ALL PASS")

/tmp/ipykernel_558/4174010939.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count_df = pd.read_sql_query(count_sql, conn)


,table_name,actual_rows,expected_rows,status
0,dim_player,157,157,PASS
1,fact_financials,204,204,PASS
2,fact_laliga_standings,11,11,PASS
3,fact_match_team_stats,631,631,PASS
4,fact_matches,631,631,PASS
5,fact_player_competition_stats,343,343,PASS
6,fact_player_match_stats,0,0,PASS
7,fact_player_season_stats,386,386,PASS
8,fact_transfers,214,214,PASS
9,fact_trophies,23,23,PASS


DATABASE ROW-COUNT CHECK: ALL PASS


## 6. Verify schemas, tables, views and load history

In [16]:
inspection_sql = '''
SELECT table_schema, table_name, table_type
FROM information_schema.tables
WHERE table_schema IN ('raw','core','audit')
ORDER BY table_schema, table_name;
'''
conn = psycopg2.connect(TARGET_DSN)
try:
    objects_df = pd.read_sql_query(inspection_sql, conn)
    load_runs_df = pd.read_sql_query(
        "SELECT * FROM audit.load_run ORDER BY load_run_id DESC",
        conn
    )
finally:
    conn.close()

print("DATABASE OBJECTS")
display(objects_df)
print("LOAD RUN HISTORY")
display(load_runs_df)

/tmp/ipykernel_558/2130408555.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  objects_df = pd.read_sql_query(inspection_sql, conn)
/tmp/ipykernel_558/2130408555.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  load_runs_df = pd.read_sql_query(


DATABASE OBJECTS


,table_schema,table_name,table_type
0,audit,data_dictionary,BASE TABLE
1,audit,data_model_grain_map,BASE TABLE
2,audit,data_quality_log,BASE TABLE
3,audit,dataset_inventory,BASE TABLE
4,audit,laliga_standings_official_validation,BASE TABLE
5,audit,load_run,BASE TABLE
6,audit,phase_4_14_validation_checks,BASE TABLE
7,audit,phase_4_15_validation_checks,BASE TABLE
8,audit,player_identity_audit,BASE TABLE
9,audit,source_register,BASE TABLE


LOAD RUN HISTORY


,load_run_id,source_release,started_at,completed_at,status,row_counts,notes
0,1,Phase 4.15 / Phase 4.16 deployment package,2026-11-19 10:44:41.642728+00:00,2026-11-19 10:44:41.642728+00:00,succeeded,"{'fact_matches': 631, 'fact_trophies': 23, 'fa...",Loaded from Phase 4.15 source CSVs; raw stagin...


In [17]:
!pg_dump --version

pg_dump (PostgreSQL) 16.15 (Ubuntu 16.15-0ubuntu0.24.04.1)


In [18]:

!apt-get update -qq
!apt-get install -y curl gnupg

!install -d /usr/share/postgresql-common/pgdg

!curl -fsSL https://www.postgresql.org/media/keys/ACCC4CF8.asc | \
  gpg --dearmor --yes -o /usr/share/postgresql-common/pgdg/apt.postgresql.org.gpg

!echo "deb [signed-by=/usr/share/postgresql-common/pgdg/apt.postgresql.org.gpg] https://apt.postgresql.org/pub/repos/apt noble-pgdg main" \
  > /etc/apt/sources.list.d/pgdg.list

!apt-get update -qq
!apt-get install -y postgresql-client-18


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
curl is already the newest version (8.5.0-2ubuntu10.16).
gnupg is already the newest version (2.4.4-2ubuntu17.6).
gnupg set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 55 not upgraded.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpq-dev libpq5
Suggested packages:
  postgresql-doc-18 libpq-oauth postgresql-18
The following NEW packages will be installed:
  postgresql-client-18
The follow

In [19]:
!pg_dump --version


pg_dump (PostgreSQL) 18.7 (Ubuntu 18.7-1.pgdg24.04+2)


In [20]:

import subprocess
from pathlib import Path
import hashlib
from getpass import getpass

# Reuse the existing connection, or request it privately
if "TARGET_DSN" not in globals():
    TARGET_DSN = getpass("Enter your Neon connection string: ")

backup_path = Path(
    "/content/Real_Madrid_Analytics_Phase_4_16.backup"
)

command = [
    "pg_dump",
    "--format=custom",
    "--no-owner",
    "--no-acl",
    "--file", str(backup_path),
    "--dbname", TARGET_DSN
]

print("Starting PostgreSQL database backup...")

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Database backup failed.")

print("BACKUP CREATED SUCCESSFULLY")

# Verify the archive can be read
check = subprocess.run(
    ["pg_restore", "--list", str(backup_path)],
    capture_output=True,
    text=True
)

if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Backup archive verification failed.")

# Calculate file size and checksum
file_size = backup_path.stat().st_size
checksum = hashlib.sha256(backup_path.read_bytes()).hexdigest()

print("Archive verification: PASS")
print(f"Backup size: {file_size:,} bytes")
print(f"SHA256: {checksum}")
print("Backup location:", backup_path)


Starting PostgreSQL database backup...
BACKUP CREATED SUCCESSFULLY
Archive verification: PASS
Backup size: 373,451 bytes
SHA256: 6a5d31c96f647234febef9f23f5cb581dd6a52864b2f3898e3a0beac1ab333d1
Backup location: /content/Real_Madrid_Analytics_Phase_4_16.backup


In [21]:
from google.colab import files

files.download(
    "/content/Real_Madrid_Analytics_Phase_4_16.backup"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Completion checklist

Do not mark Phase 4 as database-deployed until all of these are true:

- [ ] All seven deployment scripts completed without errors.
- [ ] `06_validate.sql` completed successfully.
- [ ] Every row count in the summary above is PASS.
- [ ] `raw`, `core` and `audit` objects are present.
- [ ] Load history has been reviewed.
- [ ] The database is visible in the provider dashboard.

**Security:** do not share your connection string, password, or API key. If troubleshooting is needed, share the error text after removing hostnames, usernames, passwords and other credentials.

**Do not run:** `08_reset_schemas.sql` unless you intentionally want to drop/recreate the schemas.